# 🔋 EV Battery Degradation — Industrial IoT Analysis
**Dataset:** `ev_battery_degradation_v1.csv`  
**Rows:** 10,000 vehicles · **Columns:** 13

| Column | Description |
|---|---|
| `Car_Model` | Tesla Model 3, Ford Mustang Mach-E, Hyundai Ioniq 5, Wuling Air EV, BYD Atto 3 |
| `Battery_Type` | NMC or LFP chemistry |
| `Battery_Capacity_kWh` | Rated pack size (kWh) |
| `Vehicle_Age_Months` | Vehicle age 1–96 months |
| `Total_Charging_Cycles` | Cumulative charge cycles |
| `Avg_Temperature_C` | Average operating temperature |
| `Fast_Charge_Ratio` | Fraction of fast charges (0–1) |
| `Avg_Discharge_Rate_C` | Average discharge C-rate |
| `Driving_Style` | Aggressive / Moderate / Conservative |
| `Internal_Resistance_Ohm` | Current internal resistance |
| `SoH_Percent` | **State of Health — target variable (73.7–100%)** |
| `Battery_Status` | Healthy / Replace Required |

## Step 1 — Install & Import

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn --quiet
print('✅ Done')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 12, 'axes.titleweight': 'bold'
})

MODEL_COLORS = {
    'Tesla Model 3':       '#E31937',
    'Ford Mustang Mach-E': '#003A8C',
    'Hyundai Ioniq 5':     '#00AAD4',
    'Wuling Air EV':       '#E5002D',
    'BYD Atto 3':          '#1DB954'
}
STYLE_COLORS = {'Aggressive':'#E74C3C','Moderate':'#F39C12','Conservative':'#2ECC71'}
print('✅ Libraries loaded')

## Step 2 — Upload & Profile the Data

In [ ]:
from google.colab import files
import io

print('Select ev_battery_degradation_v1.csv to upload:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'\n✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())

In [ ]:
print('Nulls:', df.isnull().sum().to_dict())
print()
display(df.describe().round(2))
print()
for col in ['Car_Model','Battery_Type','Driving_Style','Battery_Status']:
    print(f'{col}: {dict(df[col].value_counts())}')

## Step 3 — Exploratory Visualisations

Six focused plots covering all key degradation dimensions.

### Plot 1 · SoH Distribution by Car Model & Battery Chemistry

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin — SoH by Car Model
ax = axes[0]
order = df.groupby('Car_Model')['SoH_Percent'].median().sort_values().index.tolist()
sns.violinplot(data=df, x='Car_Model', y='SoH_Percent', order=order,
               palette=[MODEL_COLORS[m] for m in order], alpha=0.85, ax=ax)
ax.axhline(80, color='red', linestyle='--', lw=1.2, label='Replace threshold 80%')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.set_xlabel(''); ax.set_ylabel('SoH (%)')
ax.set_title('SoH Distribution by Car Model')
ax.legend(fontsize=9)

# Box — NMC vs LFP
ax = axes[1]
sns.boxplot(data=df, x='Battery_Type', y='SoH_Percent',
            palette={'NMC':'#3498DB','LFP':'#27AE60'}, width=0.45, ax=ax)
for btype in ['NMC','LFP']:
    m = df[df.Battery_Type == btype]['SoH_Percent'].mean()
    ax.text({'NMC':0,'LFP':1}[btype], m+0.2, f'μ={m:.1f}%',
            ha='center', fontsize=10, color='black')
ax.set_xlabel('Battery Chemistry'); ax.set_ylabel('SoH (%)')
ax.set_title('SoH by Battery Chemistry (NMC vs LFP)')

plt.suptitle('Plot 1 — State of Health: Distribution Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot1_soh_distribution.png', bbox_inches='tight')
plt.show()
print('✅ Saved plot1_soh_distribution.png')

### Plot 2 · SoH vs Primary Degradation Drivers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
drivers = [
    ('Total_Charging_Cycles', 'Total Charging Cycles'),
    ('Vehicle_Age_Months',    'Vehicle Age (Months)'),
    ('Internal_Resistance_Ohm', 'Internal Resistance (Ω)'),
]
for ax, (xcol, xlabel) in zip(axes, drivers):
    for model in df['Car_Model'].unique():
        sub = df[df.Car_Model == model]
        ax.scatter(sub[xcol], sub['SoH_Percent'],
                   s=5, alpha=0.3, color=MODEL_COLORS[model], label=model)
    # Overall trend
    z = np.polyfit(df[xcol], df['SoH_Percent'], 1)
    xs = np.linspace(df[xcol].min(), df[xcol].max(), 200)
    ax.plot(xs, np.polyval(z, xs), 'k--', lw=1.8, label='Trend')
    corr_val = df[xcol].corr(df['SoH_Percent'])
    ax.set_title(f'SoH vs {xlabel}\n(r = {corr_val:.3f})')
    ax.set_xlabel(xlabel); ax.set_ylabel('SoH (%)')
    ax.legend(markerscale=3, fontsize=7, loc='upper right')

plt.suptitle('Plot 2 — SoH vs Primary Degradation Drivers', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('plot2_soh_vs_drivers.png', bbox_inches='tight')
plt.show()
print('✅ Saved plot2_soh_vs_drivers.png')

### Plot 3 · Driving Style & Fast Charging Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped box: Driving Style × Battery Type
ax = axes[0]
order = ['Conservative','Moderate','Aggressive']
sns.boxplot(data=df, x='Driving_Style', y='SoH_Percent',
            hue='Battery_Type', order=order,
            palette={'NMC':'#3498DB','LFP':'#27AE60'}, ax=ax)
ax.set_xlabel('Driving Style'); ax.set_ylabel('SoH (%)')
ax.set_title('SoH by Driving Style & Battery Type')
ax.legend(title='Battery Type', fontsize=9)

# Fast Charge Ratio vs SoH — density hex plot
ax = axes[1]
for style in order:
    sub = df[df.Driving_Style == style]
    ax.scatter(sub['Fast_Charge_Ratio'], sub['SoH_Percent'],
               s=5, alpha=0.28, color=STYLE_COLORS[style], label=style)
# Per-style trend lines
for style in order:
    sub = df[df.Driving_Style == style]
    z = np.polyfit(sub['Fast_Charge_Ratio'], sub['SoH_Percent'], 1)
    xs = np.linspace(0, 1, 100)
    ax.plot(xs, np.polyval(z, xs), color=STYLE_COLORS[style], lw=2)
ax.set_xlabel('Fast Charge Ratio'); ax.set_ylabel('SoH (%)')
ax.set_title('SoH vs Fast Charge Ratio (by Driving Style)')
ax.legend(title='Driving Style', fontsize=9)

plt.suptitle('Plot 3 — Driving Style & Charging Behaviour', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot3_style_charging.png', bbox_inches='tight')
plt.show()
print('✅ Saved plot3_style_charging.png')

### Plot 4 · Temperature Effects

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: SoH vs Temperature coloured by Battery Type
ax = axes[0]
for btype, c in [('NMC','#3498DB'),('LFP','#27AE60')]:
    sub = df[df.Battery_Type == btype]
    ax.scatter(sub['Avg_Temperature_C'], sub['SoH_Percent'],
               s=5, alpha=0.3, color=c, label=btype)
ax.axvline(0,  color='steelblue', linestyle=':', lw=1.2, label='0°C')
ax.axvline(40, color='tomato',    linestyle=':', lw=1.2, label='40°C (heat stress)')
ax.set_xlabel('Avg Temperature (°C)'); ax.set_ylabel('SoH (%)')
ax.set_title('SoH vs Operating Temperature')
ax.legend(fontsize=9)

# Bar chart: avg SoH per temperature band
ax = axes[1]
labels = ['< 0°C','0–15°C','15–25°C','25–35°C','> 35°C']
bins = pd.cut(df['Avg_Temperature_C'], bins=[-15,0,15,25,35,55], labels=labels)
avg_soh = df.groupby(bins, observed=True)['SoH_Percent'].mean()
bar_colors = ['#3498DB','#1ABC9C','#2ECC71','#F39C12','#E74C3C']
bars = ax.bar(avg_soh.index, avg_soh.values,
              color=bar_colors, edgecolor='white', linewidth=0.8)
for b, v in zip(bars, avg_soh.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()-0.6,
            f'{v:.2f}%', ha='center', va='top', color='white',
            fontsize=10, fontweight='bold')
ax.set_xlabel('Temperature Band'); ax.set_ylabel('Average SoH (%)')
ax.set_title('Average SoH by Temperature Band')
ax.set_ylim(min(avg_soh)-1, max(avg_soh)+1)

plt.suptitle('Plot 4 — Temperature Impact on Battery Health', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot4_temperature.png', bbox_inches='tight')
plt.show()
print('✅ Saved plot4_temperature.png')

### Plot 5 · Fleet Status & Usage Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar: Replace Required % by Car Model
ax = axes[0]
counts = df.groupby(['Car_Model','Battery_Status']).size().unstack(fill_value=0)
pct = counts.div(counts.sum(axis=1), axis=0)*100
pct.plot(kind='bar', ax=ax, stacked=True,
         color={'Healthy':'#27AE60','Replace Required':'#E74C3C'},
         edgecolor='white', width=0.6)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.set_xlabel(''); ax.set_ylabel('Percentage (%)')
ax.set_title('Battery Status Breakdown by Car Model')
ax.legend(title='Status', fontsize=9)

# Pivot heatmap: avg SoH — Driving Style × Battery Type × Car Model
ax = axes[1]
pivot = df.pivot_table(values='SoH_Percent',
                       index='Driving_Style',
                       columns='Car_Model',
                       aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=92, vmax=99, ax=ax, linewidths=0.5,
            cbar_kws={'label':'Avg SoH (%)'})
ax.set_title('Avg SoH Heatmap: Driving Style × Car Model')
ax.set_xlabel(''); ax.set_ylabel('Driving Style')

plt.suptitle('Plot 5 — Fleet Status & SoH Heatmap', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot5_status_heatmap.png', bbox_inches='tight')
plt.show()
print('✅ Saved plot5_status_heatmap.png')

### Plot 6 · Full Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
num_cols = ['Battery_Capacity_kWh','Vehicle_Age_Months','Total_Charging_Cycles',
            'Avg_Temperature_C','Fast_Charge_Ratio','Avg_Discharge_Rate_C',
            'Internal_Resistance_Ohm','SoH_Percent']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink':0.8})
ax.set_title('Plot 6 — Correlation Matrix (Numeric Features)', fontweight='bold')
plt.tight_layout()
plt.savefig('plot6_correlation.png', bbox_inches='tight')
plt.show()

print('\nCorrelation with SoH_Percent (sorted):')
print(corr['SoH_Percent'].drop('SoH_Percent').sort_values().to_string())
print('✅ Saved plot6_correlation.png')

## Step 4 — Linear Regression: Feature Impact on SoH

In [ ]:
df_model = df.copy()
for col in ['Battery_Type','Driving_Style']:
    df_model[col] = LabelEncoder().fit_transform(df_model[col])

feature_cols = ['Vehicle_Age_Months','Total_Charging_Cycles','Avg_Temperature_C',
                'Fast_Charge_Ratio','Avg_Discharge_Rate_C','Internal_Resistance_Ohm',
                'Battery_Capacity_kWh','Battery_Type','Driving_Style']
X = df_model[feature_cols]
y = df_model['SoH_Percent']

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)
print(f'R²  = {r2_score(y, y_pred):.4f}')
print(f'MAE = {mean_absolute_error(y, y_pred):.4f}%')

coef_df = pd.DataFrame({'Feature':feature_cols,'Coefficient':model.coef_})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(9,5))
bar_colors = ['#E74C3C' if c<0 else '#2ECC71' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=bar_colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Regression Coefficients — Impact on SoH (%)', fontweight='bold')
ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.savefig('plot7_coefficients.png', bbox_inches='tight')
plt.show()
print(coef_df.to_string(index=False))

## Step 5 — Export for Dashboard

In [ ]:
df.to_csv('dashboard_data.csv', index=False)
print(f'✅ dashboard_data.csv — {len(df):,} rows')
from google.colab import files
files.download('dashboard_data.csv')
print('\n📥 Place dashboard_data.csv in the same folder as dashboard_app.py')
print('   pip install dash plotly pandas numpy')
print('   python dashboard_app.py')
print('   http://127.0.0.1:8050')